# 3. Model Train + SHAP / FFA Analysis

**Purpose:** Run pipeline Steps 4–8: model data → PGx → final model → SHAP → FFA/DTW/BupaR.

**Flow:** Steps 4 (model data) → 5 (PGx enrichment) → 6 (final model) → 7 (SHAP) → 8 (FFA + DTW + FP-Growth + BupaR).

**Steps:** Sync inputs → Verify → Step 4 (model data) → Step 5 (PGx) → Step 6 (final model) → Step 7 (SHAP) → Step 8 (FFA).

**Memory:** Pipeline scripts use **DuckDB and Parquet** where possible. Pandas only where required (model/SHAP APIs).

Prerequisites: Cohorts from `1_cohort_workflow.ipynb` (Step 2) and feature importance from `2_feature_importance.ipynb` plus any refined Step 3b feature lists. Run from repo root.

In [ ]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

S3_BUCKET = os.environ.get("CPIC_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("CPIC Time-to-Event Analysis - Model + SHAP/FFA Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

In [ ]:
# Both cohorts use the same production age-band set: 65-74 and 75-84.
from py_helpers.constants import REQUIRED_COHORTS

# Input dirs
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT

from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs (for Steps 4-6):")
print(f"  Cohorts (2):               {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}")
print(f"  Model data (4; in/out):    {MODEL_DATA_ROOT}")
print("\nOutput dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

## Clear all checkpoints and pipeline outputs (optional - for a fresh run)

Run this cell **once** when you want to rebuild the full pipeline from Step 4 through SHAP/FFA. It (1) clears S3 checkpoints, (2) deletes S3 pipeline outputs so Steps 4 and 6 re-run instead of re-downloading, (3) removes local output directories.

**Safety confirmation required:** edit `CONFIRM_CLEAR_ALL` in the next cell to exactly `CLEAR STEP 4-8 OUTPUTS` before running. Leave it blank to skip. After this, run the Sync cell and then Steps 4 --> 5 --> 6 --> Step 7 --> Step 8.

In [ ]:
# Clear S3 checkpoints, S3 pipeline outputs, and local outputs for a fresh model + SHAP/FFA run.
# Safety: this cell deletes generated project artifacts. It will not run unless
# CONFIRM_CLEAR_ALL exactly matches REQUIRED_CONFIRMATION.
import logging
import shutil
import subprocess
from pathlib import Path

from py_helpers.workflow_sync_checkpoint import clear_step_checkpoints, delete_step_checkpoint

REQUIRED_CONFIRMATION = "CLEAR STEP 4-8 OUTPUTS"
CONFIRM_CLEAR_ALL = ""  # Type CLEAR STEP 4-8 OUTPUTS here to enable deletion.

if CONFIRM_CLEAR_ALL != REQUIRED_CONFIRMATION:
    print("Clear-all skipped.")
    print(f"To run, set CONFIRM_CLEAR_ALL = {REQUIRED_CONFIRMATION!r} and rerun this cell.")
else:
    logger = logging.getLogger(__name__)
    print("Confirmation accepted. Clearing generated Step 4-8 checkpoints and outputs...")

    # 1) S3 checkpoint metadata so steps don't think they're done
    for step in ("4_model_data", "6_final_model"):
        for cohort, bands in REQUIRED_COHORTS.items():
            n = clear_step_checkpoints(step, cohort, bands, logger=None)
            print(f"Cleared {n} checkpoint(s) for {step} / {cohort}")

    # 2) S3 pipeline outputs so Step 4 and Step 6 re-run instead of re-downloading
    _aws = shutil.which("aws") or "aws"
    _profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []
    for prefix in ("gold/cohorts_model_data/", "gold/final_model/"):
        uri = f"s3://{S3_BUCKET}/{prefix}"
        r = subprocess.run([_aws, "s3", "rm", uri, "--recursive"] + _profile, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"Cleared S3 {uri}")
        else:
            print(f"S3 rm {uri}: exit {r.returncode}; {r.stderr or r.stdout or ''}")

    # 3) Local output directories
    dirs_to_clear = [
        MODEL_DATA_ROOT,
        FINAL_MODEL_OUTPUTS,
        FINAL_MODEL_OUTPUTS_ALT,
        PROJECT_ROOT / "7_shap_analysis" / "outputs",
        PROJECT_ROOT / "8_ffa_analysis" / "outputs",
    ]
    for d in dirs_to_clear:
        d = Path(d)
        if d.exists():
            shutil.rmtree(d)
            print(f"Removed {d}")
        else:
            print(f"(skip, not present) {d}")
    print("Done. Re-run Sync and then Steps 4 --> 5 --> 6 --> Step 7 --> Step 8 for a fresh run.")

## Sync required inputs from S3 to NVMe (idempotent)

Sync **cohorts** (Step 2), **feature importance** (Step 3/3b), and **Step 6** final model outputs from S3 so pipeline and data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync cohorts (Step 2), Step 3a/3b feature importance, and Step 6 final models from S3 to DATA_ROOT.
COHORTS_ROOT.mkdir(parents=True, exist_ok=True)
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/cohorts/", COHORTS_ROOT, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

## Step 0: Verify inputs (FI required; 4_model_data and Step 6 informational)

**Required:** **Feature importance** (Step 3/3b) — must exist for each cohort/age_band so Pipeline Step 4 can run.

**Informational:** **ModelData** checks `DATA_ROOT/4_model_data` and `PROJECT_ROOT/4_model_data` (same location `create_model_data.py` writes to). **Model** = Step 6 outputs. Both are produced by Pipeline Step 4–6 cells below; if already present, you can skip those cells.

In [ ]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    """Check if feature importance exists using FileResolver pattern."""
    from py_helpers.file_resolver import FileResolver
    # Check Step 3b refined cohort feature importance first
    resolver_3b = FileResolver(
        file_type="cohort_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    if resolver_3b.exists():
        return True
    # Fallback to Step 3a aggregated feature importance
    resolver_3a = FileResolver(
        file_type="aggregated_feature_importance",
        project_root=PROJECT_ROOT,
        cohort=cohort,
        age_band=age_band,
        auto_download=False
    )
    return resolver_3a.exists()

def check_cohorts(cohort: str, age_band: str) -> bool:
    """Check Step 2 cohort.parquet exists for at least one year (2016-2019). Layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet."""
    for year in (2016, 2017, 2018, 2019):
        p = COHORTS_ROOT / f"cohort_name={cohort}" / f"event_year={year}" / f"age_band={age_band}" / "cohort.parquet"
        if p.exists():
            return True
    return False

def check_model_data(cohort: str, age_band: str) -> bool:
    """Check model_events.parquet at canonical MODEL_DATA_ROOT (same location create_model_data.py writes to)."""
    p = MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
    return p.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/65_74/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/65-74/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)")
print("  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs")
fi_ok_all = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        cohorts_ok = check_cohorts(cohort, age_band)
        fi_ok = check_feature_importance(cohort, age_band)
        model_data_ok = check_model_data(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        if not fi_ok:
            fi_ok_all = False
        status = "ready" if fi_ok else "missing FI"
        print(f"  {cohort} / {age_band}:  Cohorts={cohorts_ok}, FI={fi_ok}, ModelData={model_data_ok}, Model={model_ok}  -> {status}")
if fi_ok_all:
    print("\nAll prerequisites are available to build model data. Run Pipeline Step 4-6 cells below.")
    print("  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)")
else:
    print("\nMissing feature importance for some cohort/age_band. Sync from S3 or run Step 3/3b first, then re-run this cell.")
if fi_ok_all:
    cohorts_missing = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands if not check_cohorts(c, ab)]
    if cohorts_missing:
        print("\nCohorts=False for some cohort/age_band. Sync gold/cohorts from S3 (run Sync cell) or run Step 2. Expected layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016-2019).")


# Pipeline Phase 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [ ]:
# Pipeline Step 4: BUILD model_events.parquet by running create_model_data.py, then QA.
# The script READS: COHORTS_ROOT (cohort.parquet), gold/medical, gold/pharmacy, and feature importance.
# It WRITES: MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet
import duckdb

FORCE_STEP4_ED = True
FORCE_STEP4_ALL = False
REBUILT_STEP4 = set()

def _model_data_candidates(cohort: str, age_band: str):
    """Canonical location for model_events.parquet (Step 4 writes to MODEL_DATA_ROOT)."""
    return [MODEL_DATA_ROOT]

def _model_data_path(cohort: str, age_band: str) -> Path:
    """Resolve model_events.parquet path (Step 4 writes to get_model_data_root() = DATA_ROOT or PROJECT on Linux)."""
    for base in _model_data_candidates(cohort, age_band):
        p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
        if p.exists():
            return p
    return None

def _log_model_data_qa(cohort: str, age_band: str) -> None:
    """Log location, target distribution, control:case ratio, and patient event-count balance."""
    path = _model_data_path(cohort, age_band)
    if not path:
        print(f"  [WARN] model_events.parquet not found for {cohort}/{age_band}")
        for base in _model_data_candidates(cohort, age_band):
            p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
            print(f"    Checked: {p}  (exists: {p.exists()})")
        print(f"    Build did not write output. Check script stdout above: [INFO] data roots and example cohort path (exists=?). Layout must be {COHORTS_ROOT}/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016-2019). Sync cohorts to COHORTS_ROOT if needed, then re-run this cell.")
        return
    print(f"  Location: {path}")
    con = duckdb.connect()
    try:
        dist = con.execute("SELECT target, COUNT(*)::BIGINT AS n FROM read_parquet(?) GROUP BY target ORDER BY target", [str(path)]).fetchall()
        total = sum(row[1] for row in dist)
        by_target = {int(row[0]): int(row[1]) for row in dist}
        n_controls = by_target.get(0, 0)
        n_cases = by_target.get(1, 0)
        ratio = (n_controls / n_cases) if n_cases else 0
        print(f"  Target distribution: {by_target} (total rows: {total:,})")
        print(f"  Control:case ratio: {n_controls:,}:{n_cases:,} = {ratio:.2f}:1")
        event_counts = con.execute(
            """
            SELECT
              target,
              COUNT(DISTINCT mi_person_key)::BIGINT AS patients,
              AVG(n_events) AS mean_events,
              MEDIAN(n_events) AS median_events,
              MIN(n_events) AS min_events,
              MAX(n_events) AS max_events
            FROM (
              SELECT mi_person_key, target, COUNT(*) AS n_events
              FROM read_parquet(?)
              GROUP BY mi_person_key, target
            )
            GROUP BY target
            ORDER BY target
            """,
            [str(path)],
        ).df()
        print("  Patient-level event-count QA:")
        print(event_counts.to_string(index=False))
    finally:
        con.close()

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Step 4: {cohort} / {age_band} (building model_events.parquet)")
        force_step4 = FORCE_STEP4_ALL or (FORCE_STEP4_ED and cohort == "ed")
        existed_before = _model_data_path(cohort, age_band) is not None
        cmd = [sys.executable, "create_model_data.py", "--cohort", cohort, "--age_band", age_band]
        if force_step4:
            cmd.append("--force-rebuild")
        r = subprocess.run(
            cmd,
            cwd=PROJECT_ROOT / "4_model_data",
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        if force_step4 or not existed_before:
            REBUILT_STEP4.add((cohort, age_band))
        _log_model_data_qa(cohort, age_band)
print("Step 4 complete.")

# Pipeline Phase 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [ ]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
# Set FORCE_STEP5 = True to re-run even when S3 outputs or checkpoints exist
FORCE_STEP5 = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Step 5: {cohort} / {age_band}")
        cmd = [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band]
        if FORCE_STEP5:
            cmd.append("--force")
        r = subprocess.run(cmd, cwd=PROJECT_ROOT / "5_pgx_analysis")
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

# Pipeline Phase 6: Final model deployment outputs

Train final models per cohort/age_band. **Default** is **per-bin** training (`--train-mode per_bin`) for both `falls` and `ed`: separate models under `outputs/.../bin_models/{low|medium|high|extreme}/`, then mirror one bin (prefer `medium`) to the cohort-level `outputs/.../{age_band}/` tree for prepare_models / Lambda. Use `--train-mode aggregate` for a single cohort-wide model only, or `both` for cohort-wide plus per-bin. Reads Step 4 model data and Step 5 PGx features; writes models and feature CSVs to `6_final_model/outputs` (or DATA_ROOT).

In [ ]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore). Default --train-mode is per_bin (omit flag).
FORCE_STEP6 = False
FORCE_STEP6_REBUILT_ONLY = True
STEP6_TRAIN_MODE = None
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"--> Step 6: {cohort} / {age_band}")
        cmd = [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band]
        if STEP6_TRAIN_MODE:
            cmd.extend(["--train-mode", STEP6_TRAIN_MODE])
        force_step6 = FORCE_STEP6 or (FORCE_STEP6_REBUILT_ONLY and (cohort, age_band) in REBUILT_STEP4)
        if force_step6:
            cmd.append("--force-retrain")
        r = subprocess.run(
            cmd,
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 6 complete.")

### Model performance per density bin and Top 20 XGBoost importance (cohort-level snapshot)

Metrics are read from `bin_models/{bin}/` (default Step 6). The cohort-level `.../{age_band}/` XGBoost FI CSV is the **mirrored deploy snapshot** (preferred bin, usually `medium`) when using `--train-mode per_bin`.


In [ ]:
# Per-bin model metrics + cohort-level FI plot (mirrored bin for deploy)
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from py_helpers.event_density_utils import DENSITY_BINS as _DENSITY_BINS

# Resolve outputs base (project or NVMe)
def _outputs_base():
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        if base and base.exists():
            return base
    return FINAL_MODEL_OUTPUTS

base = _outputs_base()
print("Final model performance - per density bin (selected model per bin)")
print("=" * 80)
all_metrics = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        for bin_name in _DENSITY_BINS:
            path = (
                base / cohort / ab_f / "bin_models" / bin_name
                / f"{cohort}_{ab_f}_model_metrics_summary.csv"
            )
            if not path.exists():
                continue
            df = pd.read_csv(path)
            selected = df.loc[df["selected"] == True]
            if selected.empty:
                selected = df.head(1)
            for _, row in selected.iterrows():
                all_metrics.append({
                    "cohort": cohort,
                    "age_band": age_band,
                    "bin": bin_name,
                    "model": row["model"],
                    "recall_mean": row["recall_mean"],
                    "pr_auc_mean": row["pr_auc_mean"],
                    "auc_mean": row.get("auc_mean", None),
                    "logloss_mean": row.get("logloss_mean", None),
                })
if all_metrics:
    summary = pd.DataFrame(all_metrics)
    print(summary.to_string(index=False))
else:
    print("  No per-bin metrics CSVs under", base / "<cohort>" / "<age_band>" / "bin_models")
print()
print("Top 20 feature importance (XGBoost) - cohort-level CSV (deploy mirror when per-bin mode)")
print("=" * 80)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        fi_path = base / cohort / ab_f / f"{cohort}_{ab_f}_xgboost_feature_importance.csv"
        if not fi_path.exists():
            print(f"  [skip] {cohort} / {age_band}: no cohort-level feature importance CSV")
            continue
        fi = pd.read_csv(fi_path).sort_values("importance", ascending=False).head(20)
        if fi.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(fi)), fi["importance"].values, align="center")
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi["feature"].values, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Importance (gain)")
        ax.set_title(f"Top 20 features - {cohort} / {age_band} (cohort-level / deploy snapshot)")
        plt.tight_layout()
        plt.show()


### Step 7: SHAP values

**7_shap_analysis/run_shap_analysis.py** — per-bin SHAP for bins trained in Step 6; falls back to cohort-level if only aggregate models exist. Run after Step 6.

In [ ]:
import logging
logger = logging.getLogger(__name__)

from py_helpers.event_density_utils import (
    cohort_aggregate_final_model_has_artifacts,
    list_trained_density_bins,
)

SHAP_SCRIPT = PROJECT_ROOT / "7_shap_analysis" / "run_shap_analysis.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        trained = list_trained_density_bins(PROJECT_ROOT, cohort, age_band)
        if trained:
            for bin_name in trained:
                print(f"--> Step 7 (SHAP bin={bin_name}): {cohort} / {age_band}")
                r = subprocess.run(
                    [sys.executable, str(SHAP_SCRIPT),
                     "--cohort", cohort, "--age_band", age_band, "--bin", bin_name],
                    cwd=PROJECT_ROOT,
                )
                if r.returncode != 0:
                    raise SystemExit(r.returncode)
        elif cohort_aggregate_final_model_has_artifacts(PROJECT_ROOT, cohort, age_band):
            print(f"--> Step 7 (SHAP cohort-level): {cohort} / {age_band}")
            r = subprocess.run(
                [sys.executable, str(SHAP_SCRIPT), "--cohort", cohort, "--age_band", age_band],
                cwd=PROJECT_ROOT,
            )
            if r.returncode != 0:
                raise SystemExit(r.returncode)
        else:
            print(f"[skip] Step 7: no Step 6 models for {cohort} / {age_band}")
print("Step 7 (SHAP) complete.")

### Step 8: FFA / DTW / FP-Growth / BupaR

**8_ffa_analysis/** — run after Step 7 (SHAP). Per-bin resolution mirrors Step 7.

In [ ]:
from py_helpers.event_density_utils import (
    cohort_aggregate_final_model_has_artifacts,
    list_trained_density_bins,
)

FFA_SCRIPT = PROJECT_ROOT / "8_ffa_analysis" / "run_ffa_workflow.py"
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        trained = list_trained_density_bins(PROJECT_ROOT, cohort, age_band)
        if trained:
            for bin_name in trained:
                print(f"--> Step 8 (FFA bin={bin_name}): {cohort} / {age_band}")
                r = subprocess.run(
                    [sys.executable, str(FFA_SCRIPT),
                     "--cohort", cohort, "--age-band", age_band, "--bin", bin_name],
                    cwd=PROJECT_ROOT,
                )
                if r.returncode != 0:
                    raise SystemExit(r.returncode)
        elif cohort_aggregate_final_model_has_artifacts(PROJECT_ROOT, cohort, age_band):
            print(f"--> Step 8 (FFA cohort-level): {cohort} / {age_band}")
            r = subprocess.run(
                [sys.executable, str(FFA_SCRIPT),
                 "--cohort", cohort, "--age-band", age_band],
                cwd=PROJECT_ROOT,
            )
            if r.returncode != 0:
                raise SystemExit(r.returncode)
        else:
            print(f"[skip] Step 8: no Step 6 models for {cohort} / {age_band}")
print("Step 8 (FFA) complete.")

### Review: Model performance summary

Run after Steps 6–8 to confirm output files exist and check per-bin metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from py_helpers.event_density_utils import DENSITY_BINS as _DENSITY_BINS

def _outputs_base():
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        if base and base.exists():
            return base
    return FINAL_MODEL_OUTPUTS

base = _outputs_base()
print("Final model performance - per density bin")
print("=" * 80)
all_metrics = []
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        for bin_name in _DENSITY_BINS:
            path = (
                base / cohort / ab_f / "bin_models" / bin_name
                / f"{cohort}_{ab_f}_model_metrics_summary.csv"
            )
            if not path.exists():
                continue
            df = pd.read_csv(path)
            selected = df.loc[df["selected"] == True]
            if selected.empty:
                selected = df.head(1)
            for _, row in selected.iterrows():
                all_metrics.append({
                    "cohort": cohort, "age_band": age_band, "bin": bin_name,
                    "model": row["model"],
                    "recall_mean": row["recall_mean"],
                    "pr_auc_mean": row["pr_auc_mean"],
                    "auc_mean": row.get("auc_mean"),
                })
if all_metrics:
    print(pd.DataFrame(all_metrics).to_string(index=False))
else:
    print(f"  No metrics CSVs found under {base}")

### Review: Top 20 XGBoost feature importance per cohort / age_band

In [ ]:
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        ab_f = age_band.replace("-", "_")
        fi_path = base / cohort / ab_f / f"{cohort}_{ab_f}_xgboost_feature_importance.csv"
        if not fi_path.exists():
            print(f"  [skip] {cohort} / {age_band}: no feature importance CSV at {fi_path}")
            continue
        fi = pd.read_csv(fi_path).sort_values("importance", ascending=False).head(20)
        if fi.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(fi)), fi["importance"].values, align="center")
        ax.set_yticks(range(len(fi)))
        ax.set_yticklabels(fi["feature"].values, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel("Importance (gain)")
        ax.set_title(f"Top 20 features - {cohort} / {age_band}")
        plt.tight_layout()
        plt.show()

### Review: SHAP output files

Confirm SHAP value parquets exist for each cohort/age_band (or per-bin).

In [ ]:
SHAP_OUTPUTS = PROJECT_ROOT / "7_shap_analysis" / "outputs"
print("SHAP outputs:")
if SHAP_OUTPUTS.exists():
    files = sorted(SHAP_OUTPUTS.rglob("*.parquet"))
    for f in files:
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(PROJECT_ROOT)}  ({size_mb:.1f} MB)")
    if not files:
        print("  (no parquet files yet - run Step 7)")
else:
    print(f"  {SHAP_OUTPUTS} does not exist - run Step 7")

# Shutdown EC2 (optional — set SHUTDOWN_EC2 = True when running on EC2)
SHUTDOWN_EC2 = False

if SHUTDOWN_EC2:
    import shutil, subprocess as _sp
    try:
        instance_id = _sp.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        if instance_id:
            aws_cmd = shutil.which("aws") or "aws"
            _profile_args = ["--profile", AWS_PROFILE] if AWS_PROFILE else []
            r = _sp.run([aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id] + _profile_args,
                        capture_output=True, text=True)
            print(f"EC2 stop {instance_id}: {'OK' if r.returncode == 0 else r.stderr.strip()}")
        else:
            print("EC2 instance ID not found — are you on EC2?")
    except Exception as e:
        print(f"EC2 shutdown skipped: {e}")
else:
    print("EC2 auto-shutdown DISABLED (SHUTDOWN_EC2 = False)")

In [ ]:
SHUTDOWN_EC2 = True  # Set to False to disable auto-shutdown

print("=" * 80)
print("Final Step: EC2 Instance Shutdown (Optional)")
print("=" * 80)

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)

    import subprocess
    import shutil
    import os

    try:
        # Retrieve EC2 instance ID from metadata service
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )

        instance_id = result.stdout.strip()

        if instance_id:
            print(f"Instance ID: {instance_id}")

            # Locate AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                for path in [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]:
                    if os.path.exists(path):
                        aws_cmd = path
                        break

            if not aws_cmd:
                print("\nWarning: AWS CLI not found. Cannot stop instance.")
                print("Install AWS CLI or ensure it is in your PATH.")
                logger.warning("AWS CLI not found; cannot stop EC2 instance")
            else:
                shutdown_cmd = [
                    aws_cmd,
                    "ec2",
                    "stop-instances",
                    "--instance-ids",
                    instance_id
                ]

                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(
                    shutdown_cmd,
                    capture_output=True,
                    text=True
                )

                if result.returncode == 0:
                    print("\nEC2 stop command sent successfully.")
                    print("Instance will stop shortly.")
                    print("Note: This is a STOP (not terminate).")
                    logger.info(
                        f"EC2 instance {instance_id} stop command issued"
                    )
                else:
                    print(
                        f"\nWarning: EC2 stop command failed "
                        f"(exit code {result.returncode})"
                    )
                    if result.stderr:
                        print(f"Error: {result.stderr.strip()}")
                    logger.warning(
                        f"EC2 stop command failed: {result.stderr}"
                    )
        else:
            print("\nWarning: Instance ID not found. Skipping shutdown.")
            print("Manual shutdown command:")
            print("  aws ec2 stop-instances --instance-ids <instance-id>")
            logger.warning("EC2 instance ID could not be determined")

    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout contacting EC2 metadata service.")
        logger.warning("Timeout retrieving EC2 instance ID")

    except Exception as e:
        print(f"\nWarning: Error during EC2 shutdown: {e}")
        logger.warning(f"EC2 shutdown exception: {e}")

else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("Set SHUTDOWN_EC2 = True to enable it.")

print("\n" + "=" * 80)
print("Workflow Complete!")
print("=" * 80)

